In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8x-seg.pt")

In [ ]:
CLASS_COLORS = {
    "cat": (255, 0, 0),        # red
    "bear": (255, 0, 0),       # red
    "giraffe": (255, 0, 0),   # red
    "sheep": (255, 0, 0),    # red
    "cow": (255, 0, 0),      # red
    "dog": (255, 0, 0),      # red
    "horse": (255, 0, 0),     # red
    "zebara": (255, 0, 0)    # red
}

### Segmentácia

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

path = r"C:\BP\pythonProject1\data_rysy\rys_trening_data_Beno\rys_trening_data_Beno"
image_paths = list(Path(path).rglob("*.jpg"))

for img_path in image_paths:
    if str(img_path).split('\\')[-1][0] < 'M':
        continue

    results = model(str(img_path), conf=0.1)
    img = cv2.imread(str(img_path))

    if img is None:
        continue

    r = results[0]

    # --- BOXES + LABELS (same as before) ---
    for box in r.boxes:
        xyxy = box.xyxy[0].cpu().numpy().astype(int)
        cls_id = int(box.cls[0].item())
        label = model.names[cls_id]
        conf = box.conf[0].item()

        cv2.rectangle(img, (xyxy[0], xyxy[1]), (xyxy[2], xyxy[3]), (0, 255, 0), 2)
        cv2.putText(img, f'{label} {conf:.2f}',
                    (xyxy[0], max(xyxy[1] - 10, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # --- MASKS (NEW PART) ---
    if r.masks is not None:
        masks = r.masks.data.cpu().numpy()  # [N, H, W]

        for i, mask in enumerate(masks):
            cls_id = int(r.boxes.cls[i].item())
            label = model.names[cls_id]

            # resize mask to image size if needed
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]))

            # color overlay
            color = CLASS_COLORS.get(label, (255,255,255))
            colored_mask = np.zeros_like(img)
            colored_mask[mask > 0.5] = color

            img = cv2.addWeighted(img, 1.0, colored_mask, 0.4, 0)

    # --- SHOW ---
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(img_path.name)
    plt.axis("off")
    plt.show()

### Ulozenie do suborovej struktury

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import os

# classes you want to keep
animal_classes = {'cow', 'sheep', 'bear', 'dog', 'cat', 'giraffe'}

src_root = Path(r"C:\BP\pythonProject1\data_rysy\rys_trening_data_Beno\rys_trening_data_Beno")
dst_root = Path(r"C:\BP\pythonProject1\data_rysy\rys_trening_data_Beno\rys_trening_data_Beno_seg")

image_paths = list(src_root.rglob("*.jpg"))

for image_path in image_paths:
    results = model(str(image_path))
    img = cv2.imread(str(image_path))

    if img is None:
        print(f"Could not load {image_path}")
        continue

    r = results[0]
    found = False

    # skip if no masks detected
    if r.masks is None:
        print(f"No segmentation masks found in {image_path}")
        continue

    masks = r.masks.data.cpu().numpy()

    for i, box in enumerate(r.boxes):
        cls_id = int(box.cls[0].item())
        label = model.names[cls_id]

        if label.lower() in animal_classes:
            mask = masks[i]

            # resize mask to original image size
            mask = cv2.resize(
                mask,
                (img.shape[1], img.shape[0]),
                interpolation=cv2.INTER_NEAREST
            )

            binary_mask = (mask > 0.5).astype(np.uint8)

            # create black background image
            segmented = np.zeros_like(img)

            # keep only segmented animal pixels
            segmented[binary_mask == 1] = img[binary_mask == 1]

            # preserve original folder structure
            rel_path = image_path.relative_to(src_root)
            save_dir = dst_root / rel_path.parent
            save_dir.mkdir(parents=True, exist_ok=True)

            save_path = save_dir / image_path.name

            cv2.imwrite(str(save_path), segmented)

            found = True
            break  # save only first valid animal mask

    if not found:
        print(f"No valid animal detected in {image_path}")